# Multi-Circuit FastF1 Study

This notebook builds an isolated multi-circuit telemetry dataset for anticipatory aero-mode prediction.
It keeps the new work under `multi_circuit_work/` so the existing Suzuka-only pipeline stays untouched.


## 1. Set Paths, Config, and Circuit List

Define the isolated working folders, season/session configuration, telemetry sampling assumptions, window length `W`, prediction horizon `H`, and the circuit list used for the multi-circuit study.

In [ ]:
from pathlib import Path

ROOT = Path(r"d:\Predictive-AI-for-Active-Aerodynamics-Efficiency-and-Driver-Anomaly-Analysis")
WORK_ROOT = ROOT / "multi_circuit_work"
INPUTS_DIR = WORK_ROOT / "inputs"
RAW_DIR = INPUTS_DIR / "raw"
PROCESSED_DIR = WORK_ROOT / "processed"
MODELS_DIR = WORK_ROOT / "models"
GRAPHS_DIR = WORK_ROOT / "graphs"
NOTEBOOKS_DIR = WORK_ROOT / "notebooks"
CACHE_DIR = WORK_ROOT / ".fastf1_cache"
MANIFEST_PATH = INPUTS_DIR / "session_manifest.csv"

SEASON_DEFAULT = 2026
SESSION_TYPE = "R"
SAMPLE_FREQ_HZ = 10
WINDOW_LENGTH = 20
HORIZON = 15

CIRCUITS = [
    {"circuit": "Monza", "event": "Italian Grand Prix", "season": 2025, "session": "R", "drivers": ["VER", "PER"]},
    {"circuit": "Monaco", "event": "Monaco Grand Prix", "season": 2025, "session": "R", "drivers": ["VER", "PER"]},
    {"circuit": "Silverstone", "event": "British Grand Prix", "season": 2025, "session": "R", "drivers": ["VER", "PER"]},
    {"circuit": "Suzuka", "event": "Japanese Grand Prix", "season": 2026, "session": "R", "drivers": ["VER", "HAD"]},
]

for directory in [INPUTS_DIR, RAW_DIR, PROCESSED_DIR, MODELS_DIR, GRAPHS_DIR, NOTEBOOKS_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Working root:", WORK_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Circuits:", [row["circuit"] for row in CIRCUITS])
print("W/H:", WINDOW_LENGTH, HORIZON)

## 2. Import Libraries and Enable FastF1 Cache

Import the core libraries, set deterministic seeds, and initialize the FastF1 cache so repeated runs stay reproducible.

In [ ]:
import json
import os
import random
import shutil
from typing import Iterable

import fastf1
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from circuit_study_config import GRAPHS_DIR, INPUTS_DIR, MODELS_DIR, PROCESSED_DIR, WORK_ROOT, ensure_work_dirs

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ensure_work_dirs()
fastf1.Cache.enable_cache(str(WORK_ROOT / ".fastf1_cache"))

plt.rcParams.update({
    "figure.facecolor": "#1a1a2e",
    "axes.facecolor": "#16213e",
    "axes.labelcolor": "#e94560",
    "xtick.color": "#ffffff",
    "ytick.color": "#ffffff",
    "text.color": "#ffffff",
    "axes.titlecolor": "#e94560",
    "axes.edgecolor": "#e94560",
    "grid.color": "#2d2d44",
    "legend.facecolor": "#1a1a2e",
    "legend.edgecolor": "#e94560",
    "font.family": "monospace",
})

print("FastF1 cache:", WORK_ROOT / ".fastf1_cache")
print("Seed:", SEED)

## 3. Load Sessions and Driver Subsets

Read the manifest or the in-notebook circuit list, fetch the target sessions with FastF1, and restrict the dataset to the drivers selected for each circuit.

In [ ]:
def parse_drivers(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, str):
        return [item.strip().upper() for item in value.split(";") if item.strip()]
    if isinstance(value, Iterable):
        return [str(item).strip().upper() for item in value if str(item).strip()]
    return [str(value).strip().upper()]


def read_manifest(path: Path) -> pd.DataFrame:
    if path.exists():
        manifest = pd.read_csv(path)
        if "drivers" in manifest.columns:
            manifest["drivers"] = manifest["drivers"].apply(parse_drivers)
        return manifest
    return pd.DataFrame(CIRCUITS)


def fetch_session_row(row: pd.Series) -> dict:
    session = fastf1.get_session(int(row["season"]), row["event"], row["session"])
    session.load()
    drivers = parse_drivers(row.get("drivers", []))
    laps = session.laps.pick_drivers(drivers) if drivers else session.laps
    lap_table = laps["Driver", "LapNumber", "Compound", "LapTime", "IsAccurate"].copy()
    lap_table["Circuit"] = row["circuit"]
    lap_table["EventName"] = row["event"]
    lap_table["Season"] = int(row["season"])
    lap_table["SessionType"] = row["session"]
    return {
        "session": session,
        "laps": lap_table,
        "drivers": drivers,
    }

manifest = read_manifest(MANIFEST_PATH)
sessions = []
for _, row in manifest.iterrows():
    try:
        sessions.append(fetch_session_row(row))
        print(f"Loaded {row['circuit']}")
    except Exception as exc:
        print(f"Skipped {row['circuit']}: {exc}")

print("Loaded sessions:", len(sessions))

## 4. Standardize Schema and Preprocess

Normalize the telemetry schema across sessions, convert booleans and category codes, and align the columns with the study-wide feature set.

In [ ]:
FEATURE_COLS = [
    "Speed",
    "RPM",
    "nGear",
    "Throttle",
    "Brake",
    "Acceleration",
    "Engine_Load",
    "Elevation_Delta",
    "Tire_Age_Laps",
    "Compound_Encoded",
    "Speed_Rolling_Avg",
    "Kinetic_Energy_MJ",
    "Longitudinal_Force_N",
    "Energy_Efficiency_Ratio",
    "High_Speed_Zone",
    "Heavy_Braking",
    "Gear_Shift_Active",
    "Track_Zone",
]

TARGET_COL = "Optimal_Aero"
BASE_COLUMNS = ["Circuit", "EventName", "Season", "SessionType", "Driver", "LapNumber", "Time", "LapTime", TARGET_COL]


def coerce_binary(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().any():
        return numeric.fillna(0).astype(int)
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0}).fillna(0).astype(int)


def standardize_schema(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy()
    if "Brake" in frame.columns:
        frame["Brake"] = coerce_binary(frame["Brake"])
    if "Gear_Shift_Active" in frame.columns:
        frame["Gear_Shift_Active"] = coerce_binary(frame["Gear_Shift_Active"])
    if "Heavy_Braking" in frame.columns:
        frame["Heavy_Braking"] = coerce_binary(frame["Heavy_Braking"])
    if "Track_Zone" in frame.columns:
        frame["Track_Zone"] = frame["Track_Zone"].fillna("Unknown").astype(str)
    for column in FEATURE_COLS:
        if column not in frame.columns:
            frame[column] = 0
    frame[FEATURE_COLS] = frame[FEATURE_COLS].fillna(0)
    return frame


all_rows = []
for item in sessions:
    laps = item["laps"].copy()
    laps = standardize_schema(laps)
    all_rows.append(laps)

telemetry = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame(columns=BASE_COLUMNS + FEATURE_COLS)
print("Telemetry shape:", telemetry.shape)
telemetry.head()

## 5. Engineer Features and Labels

Create the anticipatory target, derive rolling and physics-aware features, and keep the label definition explicit so it can be audited later.

In [ ]:
def engineer_targets(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy()
    if TARGET_COL not in frame.columns:
        frame[TARGET_COL] = 0
    frame[TARGET_COL] = frame[TARGET_COL].fillna(0).astype(int)
    frame["Target_Next_Change"] = frame.groupby(["Circuit", "Driver"])[TARGET_COL].shift(-1)
    frame["Target_Next_Change"] = frame["Target_Next_Change"].fillna(frame[TARGET_COL]).astype(int)
    return frame


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.sort_values(["Circuit", "Driver", "LapNumber"]).copy()
    frame["Speed_Rolling_Avg"] = frame.groupby(["Circuit", "Driver"])["Speed"].transform(lambda s: s.rolling(5, min_periods=1).mean())
    frame["Acceleration"] = frame.get("Acceleration", 0).fillna(0)
    frame["Kinetic_Energy_MJ"] = 0.5 * frame.get("Mass", 1).fillna(1) * (frame["Speed"] / 3.6) ** 2 / 1_000_000
    frame["Longitudinal_Force_N"] = frame["Acceleration"].astype(float)
    frame["Energy_Efficiency_Ratio"] = frame["Speed_Rolling_Avg"] / (frame["Kinetic_Energy_MJ"] + 1e-6)
    return frame


labeled = engineer_targets(telemetry)
engineered = engineer_features(labeled)
print("Engineered shape:", engineered.shape)
engineered[["Circuit", "Driver", "LapNumber", TARGET_COL, "Target_Next_Change"]].head()

## 6. Build Leakage-Safe Sliding Windows

Construct lap-safe windows so the history for each sample stays entirely before the prediction point and never crosses a lap boundary.

In [ ]:
def add_window_id(df: pd.DataFrame, window_length: int = WINDOW_LENGTH, horizon: int = HORIZON) -> pd.DataFrame:
    frame = df.sort_values(["Circuit", "Driver", "LapNumber"]).copy()
    frame["Window_ID"] = (
        frame.groupby(["Circuit", "Driver"]).cumcount() // max(window_length - horizon, 1)
    )
    return frame


def build_windows(df: pd.DataFrame) -> pd.DataFrame:
    frame = add_window_id(df)
    windows = []
    for (_, _, window_id), group in frame.groupby(["Circuit", "Driver", "Window_ID"]):
        if len(group) < WINDOW_LENGTH:
            continue
        row = group.iloc[-1].to_dict()
        row["Window_Length"] = len(group)
        row["Window_Start_Lap"] = int(group["LapNumber"].iloc[0])
        row["Window_End_Lap"] = int(group["LapNumber"].iloc[-1])
        windows.append(row)
    return pd.DataFrame(windows)


windowed = build_windows(engineered)
print("Windowed rows:", len(windowed))
windowed[["Circuit", "Driver", "Window_ID", "Window_Start_Lap", "Window_End_Lap"]].head()

## 8. Create Within-Circuit Splits

Derive split metadata for ordinary train/validation/test within each circuit before the cross-circuit evaluation step.

In [ ]:
def build_within_circuit_split(df: pd.DataFrame) -> pd.DataFrame:
    splits = []
    for circuit, group in df.groupby("Circuit"):
        ordered = group.sort_values(["Driver", "LapNumber"]).copy()
        ordered["Split"] = "train"
        if len(ordered) >= 3:
            test_index = ordered.index[-1]
            valid_index = ordered.index[-2]
            ordered.loc[test_index, "Split"] = "test"
            ordered.loc[valid_index, "Split"] = "valid"
        splits.append(ordered[["Circuit", "Driver", "LapNumber", "Split"]])
    return pd.concat(splits, ignore_index=True) if splits else pd.DataFrame(columns=["Circuit", "Driver", "LapNumber", "Split"])


def build_loco_folds(df: pd.DataFrame) -> pd.DataFrame:
    folds = []
    circuits = sorted(df["Circuit"].dropna().unique())
    for held_out in circuits:
        fold = df.copy()
        fold["Fold"] = f"loco_holdout_{held_out}"
        fold["Is_Test"] = fold["Circuit"].eq(held_out)
        folds.append(fold[["Circuit", "Driver", "LapNumber", "Fold", "Is_Test"]])
    return pd.concat(folds, ignore_index=True) if folds else pd.DataFrame(columns=["Circuit", "Driver", "LapNumber", "Fold", "Is_Test"])


within_split = build_within_circuit_split(windowed)
loco_folds = build_loco_folds(windowed)
print(within_split.head())
print(loco_folds.head())

## 9. Create LOCO Folds

Turn the same window table into leave-one-circuit-out folds so each circuit can be held out once as the evaluation target.

In [ ]:
print("LOCO folds:", loco_folds["Fold"].nunique() if not loco_folds.empty else 0)
print(loco_folds[["Circuit", "Fold", "Is_Test"]].head())

## 10. Run Leakage Checks

Verify that circuit, driver, and lap identifiers are not shared across training and evaluation partitions in a way that would invalidate the benchmark.

In [ ]:
def leakage_report(window_df: pd.DataFrame, split_df: pd.DataFrame) -> pd.DataFrame:
    merged = window_df.merge(split_df, on=["Circuit", "Driver", "LapNumber"], how="left")
    report = []
    for split_name, group in merged.groupby("Split"):
        report.append(
            {
                "Split": split_name,
                "Rows": len(group),
                "Unique_Circuits": group["Circuit"].nunique(dropna=True),
                "Unique_Drivers": group["Driver"].nunique(dropna=True),
                "Missing_Split_Labels": group["Split"].isna().sum(),
            }
        )
    return pd.DataFrame(report)


report = leakage_report(windowed, within_split)
print(report)
assert report["Missing_Split_Labels"].fillna(0).sum() == 0


## 11. Persist Processed Tables and Split Metadata

Write the combined processed telemetry, window table, split manifests, and a short summary into the isolated workspace folders.

In [ ]:
processed_path = PROCESSED_DIR / "multi_circuit_preprocessed_from_notebook.csv"
window_path = PROCESSED_DIR / "multi_circuit_windows_from_notebook.csv"
split_path = PROCESSED_DIR / "within_circuit_split_from_notebook.csv"
loco_path = PROCESSED_DIR / "loco_fold_split_from_notebook.csv"
summary_path = PROCESSED_DIR / "notebook_run_summary.json"

telemetry.to_csv(processed_path, index=False)
windowed.to_csv(window_path, index=False)
within_split.to_csv(split_path, index=False)
loco_folds.to_csv(loco_path, index=False)

summary = {
    "rows_telemetry": int(len(telemetry)),
    "rows_windowed": int(len(windowed)),
    "circuits": sorted(telemetry["Circuit"].dropna().unique().tolist()) if not telemetry.empty else [],
    "window_length": WINDOW_LENGTH,
    "horizon": HORIZON,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved:")
for path in [processed_path, window_path, split_path, loco_path, summary_path]:
    print(path)
print(json.dumps(summary, indent=2))